# BM25 Retriever Tests
Small, deterministic checks for BM25 behavior on a toy corpus.

In [3]:
import os
import sys
import pandas as pd
import nltk

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

project_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_dir = os.path.join(project_dir, "data")
utils_dir = os.path.join(project_dir, "source", "utils")

for path in (project_dir, utils_dir):
    if path not in sys.path:
        sys.path.append(path)

from source.utils.textPreprocessing import TextPreprocessing
from source.utils.inverted_index import InvertedIndex
from source.utils.BM25_retriever import BM25Retriever

documents_path = os.path.join(data_dir, "documents.csv")
queries_path = os.path.join(data_dir, "queries.csv")

documents = pd.read_csv(documents_path)
queries = pd.read_csv(queries_path)

tp = TextPreprocessing()
processed = tp.process_dataframe(documents)
vocab = tp.get_vocab()

index = InvertedIndex(processed, vocab)
index._build()

bm25 = BM25Retriever(processed, index)

In [11]:
# Basic checks on real dataset
assert len(documents) > 0
assert len(queries) > 0

# Run a few sample queries
sample_queries = queries.head(5).to_dict(orient="records")
results = []
for row in sample_queries:
    qid = row["id"]
    query_text = row["content"]
    hits = bm25.search(query_text, top_k=30)
    results.append({"qid": qid, "query": query_text, "top5": hits})

results_df = pd.DataFrame(results)
results_df

,qid,query,top5
0,1,what similarity laws must be obeyed when cons...,"[(486, 19.58115926640203), (184, 19.5145965396..."
1,2,what are the structural and aeroelastic probl...,"[(12, 28.473374692881826), (51, 17.10420840110..."
2,3,what problems of heat conduction in composite...,"[(485, 25.601391974150093), (5, 23.94330880137..."
3,4,can a criterion be developed to show empirica...,"[(488, 28.43412484048587), (166, 27.5491265873..."
4,5,what chemical kinetic system is applicable to...,"[(103, 15.986925419436375), (1032, 15.04532492..."


# Evaluation Metrics (REL Ground Truth)
Compute MAP@10, NDCG@10, Precision/Recall/F1 at 10 using Cranfield REL files.

In [12]:
import math
import numpy as np

rel_dir = os.path.abspath(os.path.join(project_dir, "..", "Dataset", "Cranfield", "REL"))

def load_relations(rel_folder):
    qrels = {}
    for name in os.listdir(rel_folder):
        if not name.endswith(".txt"):
            continue
        path = os.path.join(rel_folder, name)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                qid = int(parts[0])
                doc_id = int(parts[1])
                rel = float(parts[2])
                qrels.setdefault(qid, {})[doc_id] = rel
    return qrels

def normalize_relevance(rel_map):
    positive = [rel for rel in rel_map.values() if rel > 0]
    if not positive:
        return {}
    max_rel = max(positive)
    return {doc_id: (max_rel - rel + 1) if rel > 0 else 0.0 for doc_id, rel in rel_map.items()}

def precision_recall_f1_at_k(retrieved, relevant_set, k):
    if k <= 0:
        return 0.0, 0.0, 0.0
    top_k = retrieved[:k]
    if not top_k:
        return 0.0, 0.0, 0.0
    hit = sum(1 for doc_id in top_k if doc_id in relevant_set)
    precision = hit / k
    recall = hit / max(len(relevant_set), 1)
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

def average_precision_at_k(retrieved, relevant_set, k):
    if not relevant_set:
        return 0.0
    ap_sum = 0.0
    hit = 0
    for i, doc_id in enumerate(retrieved[:k], start=1):
        if doc_id in relevant_set:
            hit += 1
            ap_sum += hit / i
    return ap_sum / len(relevant_set)

def ndcg_at_k(retrieved, rel_map, k):
    def dcg(scores):
        return sum((2 ** rel - 1) / math.log2(i + 2) for i, rel in enumerate(scores))
    gains = [rel_map.get(doc_id, 0.0) for doc_id in retrieved[:k]]
    ideal = sorted(rel_map.values(), reverse=True)[:k]
    dcg_val = dcg(gains)
    idcg_val = dcg(ideal)
    if idcg_val == 0:
        return 0.0
    return dcg_val / idcg_val

qrels = load_relations(rel_dir)

k = 25
precision_list = []
recall_list = []
f1_list = []
ap_list = []
ndcg_list = []

for row in queries.to_dict(orient="records"):
    qid = int(row["id"])
    query_text = row["content"]
    hits = bm25.search(query_text, top_k=k)
    retrieved = [doc_id for doc_id, _ in hits]
    rel_map_raw = qrels.get(qid, {})
    rel_map = normalize_relevance(rel_map_raw)
    relevant_set = {doc_id for doc_id, rel in rel_map_raw.items() if rel > 0}

    precision, recall, f1 = precision_recall_f1_at_k(retrieved, relevant_set, k)
    ap = average_precision_at_k(retrieved, relevant_set, k)
    ndcg = ndcg_at_k(retrieved, rel_map, k)

    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)
    ap_list.append(ap)
    ndcg_list.append(ndcg)

metrics = {
    f"MAP@{k}": float(np.mean(ap_list)) if ap_list else 0.0,
    f"NDCG@{k}": float(np.mean(ndcg_list)) if ndcg_list else 0.0,
    f"Precision@{k}": float(np.mean(precision_list)) if precision_list else 0.0,
    f"Recall@{k}": float(np.mean(recall_list)) if recall_list else 0.0,
    f"F1@{k}": float(np.mean(f1_list)) if f1_list else 0.0,
}

metrics

{'MAP@25': 0.2671214004131747,
 'NDCG@25': 0.3844697682712288,
 'Precision@25': 0.1368888888888889,
 'Recall@25': 0.5372652072729471,
 'F1@25': 0.20329746513293717}